# Round 1 Benchmarking — Test 1a: Attribute Accuracy

Predict all **5 discourse-context micro-tags** for each sentence across multiple LLMs.
Each attribute is tested in its own cell using a structured zero-shot prompt.

| Setting | Value |
|---|---|
| Dataset | GOLD_187.csv |
| Task | Attribute Accuracy (Test 1a) |
| Models | GPT, Claude, Gemini, DeepSeek, SEA-LION (+ optional local Llama) |
| Attributes | Epistemic_Stance, Particle_Position, Listener_Agreement, Emotion, Question_Type |

Test the most and least powerful models on this task, 

GPT (general ): [REDACTED]

Claude: [REDACTED]

Gemini: [REDACTED]
(如果上面的gemini api到了限度，可以用这个) [REDACTED]

DeepSeek: [REDACTED]
Llama: 1b, 3b and 7b insturction-tuned models 
SEA-LION: [REDACTED]


In [4]:
import json
import os
import re
import time
from pathlib import Path

import anthropic
import google.generativeai as genai
import pandas as pd
import requests
from openai import OpenAI
from tqdm import tqdm

pd.set_option("display.max_colwidth", 120)


def _extract_keys_from_notebook(nb_path: Path):
    """Parse markdown notes cell for API keys."""
    keys = {
        "openai": None,
        "anthropic": None,
        "gemini": [],
        "deepseek": None,
        "sealion": None,
    }
    try:
        nb = json.loads(nb_path.read_text(encoding="utf-8"))
        text = "\n".join(
            "\n".join(cell.get("source", []))
            for cell in nb.get("cells", [])
            if cell.get("cell_type") == "markdown"
        )
        patterns = {
            "openai": r"GPT\s*\(general\s*\)\s*:\s*(sk-[A-Za-z0-9_\-]+)",
            "anthropic": r"Claude\s*:\s*(sk-ant-[A-Za-z0-9_\-]+)",
            "gemini": r"Gemini\s*:\s*([A-Za-z0-9_\-]+)",
            "deepseek": r"DeepSeek\s*:\s*(sk-[A-Za-z0-9_\-]+)",
            "sealion": r"SEA-LION\s*:\s*(sk-[A-Za-z0-9_\-]+)",
        }
        for k, pat in patterns.items():
            m = re.search(pat, text)
            if m and k != "gemini":
                keys[k] = m.group(1).strip()
        keys["gemini"] = re.findall(r"AIza[0-9A-Za-z_\-]+", text)
    except Exception:
        pass
    return keys


# ── Keys and clients ──────────────────────────────────────────────────────────
notebook_path = Path("04_round1_1a_attribute_accuracy.ipynb")
fallback = _extract_keys_from_notebook(notebook_path)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or fallback["openai"]
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY") or fallback["anthropic"]
GEMINI_API_KEYS = [key for key in [os.getenv("GEMINI_API_KEY")] + fallback["gemini"] if key]
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY") or fallback["deepseek"]
SEA_LION_API_KEY = os.getenv("SEA_LION_API_KEY") or fallback["sealion"]

# Optional runtime for local llama via Ollama.
LLAMA_BASE_URL = os.getenv("LLAMA_BASE_URL", "http://localhost:11434")
LLAMA_STRONG_MODEL = os.getenv("LLAMA_STRONG_MODEL", "llama3.2:7b")
LLAMA_WEAK_MODEL = os.getenv("LLAMA_WEAK_MODEL", "llama3.2:1b")

# Optional SEA-LION endpoint if OpenAI-compatible.
SEA_LION_BASE_URL = os.getenv("SEA_LION_BASE_URL", "https://api.sea-lion.ai/v1")
SEA_LION_STRONG_MODEL = os.getenv("SEA_LION_STRONG_MODEL", "aisingapore/Llama-SEA-LION-v3.5-70B-R")
SEA_LION_WEAK_MODEL = os.getenv("SEA_LION_WEAK_MODEL", "aisingapore/Gemma-SEA-LION-v4-27B-IT")
GPT_STRONG_MODEL = os.getenv("GPT_STRONG_MODEL", "gpt-5.5")
GPT_WEAK_MODEL = os.getenv("GPT_WEAK_MODEL", "gpt-5.5-instant")
DEEPSEEK_STRONG_MODEL = os.getenv("DEEPSEEK_STRONG_MODEL", "deepseek-v4-pro")
DEEPSEEK_WEAK_MODEL = os.getenv("DEEPSEEK_WEAK_MODEL", "deepseek-v4-flash")
CLAUDE_STRONG_MODEL = os.getenv("CLAUDE_STRONG_MODEL", "claude-opus-4-7")
CLAUDE_WEAK_MODEL = os.getenv("CLAUDE_WEAK_MODEL", "claude-4.6-haiku")
GEMINI_STRONG_MODEL = os.getenv("GEMINI_STRONG_MODEL", "models/gemini-3.1-pro-preview")
GEMINI_WEAK_MODEL = os.getenv("GEMINI_WEAK_MODEL", "models/gemini-3.1-flash-lite")

MODEL_RUNS = []

if OPENAI_API_KEY:
    MODEL_RUNS.append({
        "name": "gpt_strong",
        "provider": "openai",
        "client": OpenAI(api_key=OPENAI_API_KEY),
        "model": GPT_STRONG_MODEL,
        "sleep": 0.5,
    })
    MODEL_RUNS.append({
        "name": "gpt_weak",
        "provider": "openai",
        "client": OpenAI(api_key=OPENAI_API_KEY),
        "model": GPT_WEAK_MODEL,
        "sleep": 0.5,
    })

if ANTHROPIC_API_KEY:
    MODEL_RUNS.append({
        "name": "claude_strong",
        "provider": "anthropic",
        "client": anthropic.Anthropic(api_key=ANTHROPIC_API_KEY),
        "model": CLAUDE_STRONG_MODEL,
        "sleep": 0.5,
    })
    MODEL_RUNS.append({
        "name": "claude_weak",
        "provider": "anthropic",
        "client": anthropic.Anthropic(api_key=ANTHROPIC_API_KEY),
        "model": CLAUDE_WEAK_MODEL,
        "sleep": 0.5,
    })

if GEMINI_API_KEYS:
    MODEL_RUNS.append({
        "name": "gemini_strong",
        "provider": "gemini",
        "client": None,
        "api_keys": GEMINI_API_KEYS,
        "model": GEMINI_STRONG_MODEL,
        "sleep": 0.5,
    })
    MODEL_RUNS.append({
        "name": "gemini_weak",
        "provider": "gemini",
        "client": None,
        "api_keys": GEMINI_API_KEYS,
        "model": GEMINI_WEAK_MODEL,
        "sleep": 0.5,
    })

if DEEPSEEK_API_KEY:
    MODEL_RUNS.append({
        "name": "deepseek_strong",
        "provider": "openai",
        "client": OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com"),
        "model": DEEPSEEK_STRONG_MODEL,
        "sleep": 0.3,
    })
    MODEL_RUNS.append({
        "name": "deepseek_weak",
        "provider": "openai",
        "client": OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com"),
        "model": DEEPSEEK_WEAK_MODEL,
        "sleep": 0.3,
    })

llama_available = False
try:
    tags_url = f"{LLAMA_BASE_URL.rstrip('/')}/api/tags"
    resp = requests.get(tags_url, timeout=2)
    llama_available = resp.ok
except Exception:
    llama_available = False

if llama_available:
    MODEL_RUNS.append({
        "name": "llama_strong",
        "provider": "ollama",
        "client": None,
        "model": LLAMA_STRONG_MODEL,
        "sleep": 0.2,
        "base_url": LLAMA_BASE_URL,
    })
    MODEL_RUNS.append({
        "name": "llama_weak",
        "provider": "ollama",
        "client": None,
        "model": LLAMA_WEAK_MODEL,
        "sleep": 0.2,
        "base_url": LLAMA_BASE_URL,
    })
else:
    print("Llama skipped: Ollama server not reachable at", LLAMA_BASE_URL)

if SEA_LION_API_KEY:
    MODEL_RUNS.append({
        "name": "sealion_strong",
        "provider": "openai",
        "client": OpenAI(api_key=SEA_LION_API_KEY, base_url=SEA_LION_BASE_URL),
        "model": SEA_LION_STRONG_MODEL,
        "sleep": 0.3,
    })
    MODEL_RUNS.append({
        "name": "sealion_weak",
        "provider": "openai",
        "client": OpenAI(api_key=SEA_LION_API_KEY, base_url=SEA_LION_BASE_URL),
        "model": SEA_LION_WEAK_MODEL,
        "sleep": 0.3,
    })

if not MODEL_RUNS:
    raise ValueError("No model keys/runtimes configured.")

print("Models to run:", [m["name"] for m in MODEL_RUNS])

# ── Load data ─────────────────────────────────────────────────────────────────
DATA_PATH = Path("../Datasets/GOLD_187.csv")
df = pd.read_csv(DATA_PATH)

# Pilot mode: first run on 5 samples, then set to None for full dataset.
SAMPLE_N = 5
eval_df = df.head(SAMPLE_N).copy() if SAMPLE_N else df.copy()
results = eval_df.copy()

print(f"Loaded {len(df)} total rows; evaluating {len(eval_df)} rows")
display(eval_df.head(3))

Llama skipped: Ollama server not reachable at http://localhost:11434
Models to run: ['gpt_strong', 'gpt_weak', 'claude_strong', 'claude_weak', 'gemini_strong', 'gemini_weak', 'deepseek_strong', 'deepseek_weak', 'sealion_strong', 'sealion_weak']
Loaded 187 total rows; evaluating 5 rows


,Text,Source,Particle,Sentence_Type,Epistemic_Stance,Listener_Agreement,Emotion,Question_Type,Particle_Position
0,Menggatal/Tuaran no electric since 8:50pm. Ada ke notis dari tadi sy check fb sesb tda notis. Tu karen kan cukup2 ut...,Reddit,ke,Natural,Certain,Neutral/Unclear,Neutral/Unclear,Rhetorical Interrogative,Middle/End
1,Like tak pelik ke nape aku eja lain macam. Dia cakap bahasa wechat. Tu kan nama aku diterbalikkan 😭😂,Reddit,kan,Natural,Certain,Assumed Agreement,Negative,Rhetorical Interrogative,Middle/End
2,Like tak pelik ke nape aku eja lain macam. Dia cakap bahasa wechat. Tu [___] nama aku diterbalikkan 😭😂,Reddit,neutral,Synthesised (removed kan),Certain,Neutral/Unclear,Negative,Declarative/Statement,NaN


In [7]:
import requests


def _extract_label(raw, label_set):
    text = str(raw or "").strip()
    for label in sorted(label_set, key=len, reverse=True):
        if label.lower() in text.lower():
            return label
    return text


def _is_fatal_error(error):
    text = str(error).lower()
    fatal_markers = [
        "model_not_found",
        "does not exist",
        "unsupported parameter",
        "unsupported value",
        "invalid_request_error",
        "api key not valid",
        "permission",
        "authentication",
        "deadline expired",
        "service unavailable",
        "high demand",
    ]
    return any(marker in text for marker in fatal_markers)


def _call_gemini_with_fallback(run_cfg, prompt_text):
    last_error = None
    for api_key in run_cfg.get("api_keys", []):
        try:
            genai.configure(api_key=api_key)
            client = genai.GenerativeModel(run_cfg["model"])
            response = client.generate_content(
                prompt_text,
                generation_config={"temperature": 0, "max_output_tokens": 12},
                request_options={"timeout": 15},
            )
            return (response.text or "").strip()
        except Exception as error:
            last_error = error
            if _is_fatal_error(error):
                break
            continue
    raise last_error


def call_llm(run_cfg, prompt_text, label_set, retries=3, delay=1.0):
    """Call one configured model and return the best-matching label from label_set."""
    constraint = "You must output exactly one label from the provided set and nothing else."

    for attempt in range(retries):
        try:
            provider = run_cfg["provider"]
            model = run_cfg["model"]

            if provider == "openai":
                response = run_cfg["client"].chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": constraint},
                        {"role": "user", "content": prompt_text},
                    ],
                    max_completion_tokens=12,
                )
                raw = response.choices[0].message.content.strip()

            elif provider == "anthropic":
                response = run_cfg["client"].messages.create(
                    model=model,
                    system=constraint,
                    max_tokens=12,
                    messages=[{"role": "user", "content": prompt_text}],
                )
                raw = response.content[0].text.strip()

            elif provider == "gemini":
                raw = _call_gemini_with_fallback(run_cfg, f"{constraint}\n\n{prompt_text}")

            elif provider == "ollama":
                url = f"{run_cfg['base_url'].rstrip('/')}/api/generate"
                response = requests.post(
                    url,
                    json={
                        "model": model,
                        "prompt": f"{constraint}\n\n{prompt_text}",
                        "stream": False,
                        "options": {"temperature": 0},
                    },
                    timeout=60,
                )
                response.raise_for_status()
                raw = response.json().get("response", "").strip()

            else:
                raise ValueError(f"Unsupported provider: {provider}")

            return _extract_label(raw, label_set)

        except Exception as e:
            if _is_fatal_error(e):
                return f"ERROR_FATAL: {e}"
            if attempt < retries - 1:
                time.sleep(delay * (attempt + 1))
            else:
                return f"ERROR: {e}"


def run_attribute(attribute, label_set, prompt_template, col_suffix=""):
    """Run all configured models for one attribute and append prediction columns to `results`."""
    print(f"\n{'─'*60}")
    print(f"  Attribute : {attribute}")
    print(f"  Labels    : {label_set}")
    if col_suffix:
        print(f"  Variant   : {col_suffix}")
    print(f"{'─'*60}")

    strict_tail = "\n\nReturn exactly one label from this set and nothing else: " + ", ".join(label_set)

    for run_cfg in MODEL_RUNS:
        model_name = run_cfg["name"]
        preds = []
        skip_model = False
        cached_fatal = None
        for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=f"{model_name.upper():>8}"):
            if skip_model:
                preds.append(cached_fatal)
                continue
            prompt = prompt_template.format(TEXT=row["Text"]) + strict_tail
            pred = call_llm(run_cfg, prompt, label_set)
            preds.append(pred)
            if isinstance(pred, str) and pred.startswith("ERROR_FATAL:"):
                skip_model = True
                cached_fatal = pred
            time.sleep(run_cfg["sleep"])
        col = f"{model_name}_{attribute}{col_suffix}"
        results[col] = preds
        print(f"  {model_name.upper()} done → sample: {preds[:3]}")

## Attribute 1 — Epistemic Stance
How certain does the speaker sound about the information they are conveying?

In [8]:
EPISTEMIC_LABELS = ["Certain", "Uncertain", "Neutral/Unclear", "Neutral / NA"]

EPISTEMIC_PROMPT = """\
You are a linguist specialising in colloquial Malay. Your task is to read the Malay sentence below and decide how certain the speaker sounds about the information they are conveying.
Referring to the following three labels and their definitions to make your decision:
Certain: The speaker treats the statement as already true or established. There is no hedging, no doubt, and no checking. The speaker is asserting the information with full confidence.
Uncertain: The speaker sounds unsure, is making a guess, is estimating, or is checking whether something is the case. Words like "agaknya" (I think/probably), "kot" (maybe), or question particles that probe for confirmation are typical signals.
Neutral/NA: The sentence does not carry any detectable certainty signal in either direction. This applies to neutral descriptions, commands, or sentences where certainty is simply not relevant.
Speaker: "{TEXT}"
Given the three labels "Certain, Uncertain, Neutral/NA", the most likely, single label of the speaker's utterance is:"""

run_attribute("Epistemic_Stance", EPISTEMIC_LABELS, EPISTEMIC_PROMPT)


────────────────────────────────────────────────────────────
  Attribute : Epistemic_Stance
  Labels    : ['Certain', 'Uncertain', 'Neutral/Unclear', 'Neutral / NA']
────────────────────────────────────────────────────────────


GPT_STRONG: 100%|██████████| 5/5 [00:09<00:00,  1.85s/it]


  GPT_STRONG done → sample: ['', '', '']


GPT_WEAK: 100%|██████████| 5/5 [00:00<00:00,  7.76it/s]


  GPT_WEAK done → sample: ["ERROR_FATAL: Error code: 404 - {'error': {'message': 'The model `gpt-5.5-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}", "ERROR_FATAL: Error code: 404 - {'error': {'message': 'The model `gpt-5.5-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}", "ERROR_FATAL: Error code: 404 - {'error': {'message': 'The model `gpt-5.5-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}"]


...

  DEEPSEEK_WEAK done → sample: ['Uncertain', 'Uncertain', 'Certain']


SEALION_STRONG: 100%|██████████| 5/5 [00:05<00:00,  1.07s/it]


  SEALION_STRONG done → sample: ["Okay, let's tackle this. The user is asking me", "Okay, let's tackle this. The user wants me to", "Okay, let's tackle this problem. The user wants me"]


SEALION_WEAK: 100%|██████████| 5/5 [00:04<00:00,  1.20it/s]

  SEALION_WEAK done → sample: ['Uncertain', 'Uncertain', 'Uncertain']


## Attribute 2 — Particle Position
Where does the particle appear in the sentence?

In [ ]:
PARTICLE_POSITION_LABELS = ["Front", "Middle/End", "N/A"]

PARTICLE_POSITION_PROMPT = """\
You are a linguist specialising in colloquial Malay. Your task is to read the Malay sentence below and decide where the discourse particle appears in it.
Referring to the following three labels and their definitions to make your decision:
Front: The particle appears at the very start of the sentence, before any other content words.
Middle/End: The particle appears anywhere other than the front — mid-sentence, before the final word, or at the end.
N/A: No discourse particle is present in the sentence (e.g. the particle slot is shown as "[___]" or the sentence simply contains no particle).
Speaker: "{TEXT}"
Given the three labels "Front, Middle/End, N/A", the most likely, single label of the speaker's utterance is:"""

run_attribute("Particle_Position", PARTICLE_POSITION_LABELS, PARTICLE_POSITION_PROMPT)


────────────────────────────────────────────────────────────
  Attribute : Particle_Position
  Labels    : ['Front', 'Middle/End', 'N/A']
────────────────────────────────────────────────────────────


GPT_STRONG: 100%|██████████| 5/5 [00:08<00:00,  1.74s/it]


  GPT_STRONG done → sample: ['', '', '']


GPT_WEAK: 100%|██████████| 5/5 [00:00<00:00,  5.65it/s]


  GPT_WEAK done → sample: ["ERROR_FATAL: Error code: 404 - {'error': {'message': 'The model `gpt-5.5-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}", "ERROR_FATAL: Error code: 404 - {'error': {'message': 'The model `gpt-5.5-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}", "ERROR_FATAL: Error code: 404 - {'error': {'message': 'The model `gpt-5.5-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}"]


CLAUDE_STRONG: 100%|██████████| 5/5 [00:09<00:00,  1.97s/it]


  CLAUDE_STRONG done → sample: ['Middle/End', 'Middle/End', 'N/A']


CLAUDE_WEAK: 100%|██████████| 5/5 [00:19<00:00,  3.89s/it]


  CLAUDE_WEAK done → sample: ["ERROR: Error code: 404 - {'type': 'error', 'error': {'type': 'not_found_error', 'message': 'model: claude-4.6-haiku'}, 'request_id': 'req_011CazxGmVEfwUCaVUwFV5ia'}", "ERROR: Error code: 404 - {'type': 'error', 'error': {'type': 'not_found_error', 'message': 'model: claude-4.6-haiku'}, 'request_id': 'req_011CazxH3gZyPgssnyj5Wr3F'}", "ERROR: Error code: 404 - {'type': 'error', 'error': {'type': 'not_found_error', 'message': 'model: claude-4.6-haiku'}, 'request_id': 'req_011CazxHMjmHFqGdJ8MES9dB'}"]


GEMINI_STRONG:   0%|          | 0/5 [00:00<?, ?it/s]

## Attribute 3 — Listener Agreement
How is the speaker orienting toward the listener in terms of shared knowledge or agreement?

In [12]:
LISTENER_LABELS = ["Assumed Agreement", "Confirmation Seeking", "Neutral/Unclear"]

LISTENER_PROMPT = """\
You are a linguist specialising in colloquial Malay. Your task is to read the Malay sentence below and decide how the speaker is orienting toward the listener in terms of shared knowledge or agreement.
Referring to the following three labels and their definitions to make your decision:
Assumed Agreement: The speaker treats the information as already shared or obvious to the listener. The sentence is presented as common ground — the underlying tone is "you already know this" or "of course this is true". No explicit confirmation is being requested.
Confirmation Seeking: The speaker is actively checking whether the listener agrees, knows, or can confirm the information. The sentence invites or requests the listener's validation before the speaker can proceed with confidence.
Neutral/Unclear: The sentence does not show any clear orientation toward listener agreement. This applies to plain statements, commands, or cases where the interpersonal stance toward agreement is ambiguous or absent.
Speaker: "{TEXT}"
Given the three labels "Assumed Agreement, Confirmation Seeking, Neutral/Unclear", the most likely, single label of the speaker's utterance is:"""

run_attribute("Listener_Agreement", LISTENER_LABELS, LISTENER_PROMPT)


────────────────────────────────────────────────────────────
  Attribute : Listener_Agreement
  Labels    : ['Assumed Agreement', 'Confirmation Seeking', 'Neutral/Unclear']
────────────────────────────────────────────────────────────


     GPT: 100%|██████████| 5/5 [00:05<00:00,  1.02s/it]


  GPT done → sample: ['Confirmation Seeking', 'Confirmation Seeking', 'Confirmation Seeking']


  CLAUDE: 100%|██████████| 5/5 [00:07<00:00,  1.58s/it]


  CLAUDE done → sample: ['Confirmation Seeking', 'Assumed Agreement', 'Confirmation Seeking']


...

  GEMINI done → sample: ['ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\nPlease retry in 21.891123198s. [links {\n  description: "Learn more about Gemini API quotas"\n  url: "https://ai.google.dev/gemini-api/docs/rate-limits"\n}\n, violations {

DEEPSEEK: 100%|██████████| 5/5 [00:10<00:00,  2.08s/it]


  DEEPSEEK done → sample: ['Confirmation Seeking', 'Confirmation Seeking', 'Confirmation Seeking']


 SEALION: 100%|██████████| 5/5 [00:05<00:00,  1.04s/it]

  SEALION done → sample: ["Okay, let's tackle this. The user wants me to", "Okay, let's break down this Malay sentence. The speaker", "Okay, let's tackle this. The user wants me to"]


## Attribute 4 — Emotion
What is the emotional tone of the speaker's utterance?

In [13]:
EMOTION_LABELS = ["Positive", "Negative", "Neutral/Unclear"]

EMOTION_PROMPT = """\
You are a linguist specialising in colloquial Malay. Your task is to read the Malay sentence below and decide the emotional tone of the speaker.
Referring to the following three labels and their definitions to make your decision:
Positive: The speaker expresses happiness, excitement, enthusiasm, satisfaction, humour, affection, relief, or any other clearly positive feeling. This includes light-hearted teasing or playful sarcasm that is warm in tone.
Negative: The speaker expresses frustration, annoyance, disappointment, sadness, anger, bitterness, or any other clearly negative feeling. This includes hostile or bitter sarcasm.
Neutral/Unclear: The sentence carries no detectable emotional charge in either direction, or the emotion is genuinely ambiguous and cannot be reliably classified as positive or negative.
Speaker: "{TEXT}"
Given the three labels "Positive, Negative, Neutral/Unclear", the most likely, single label of the speaker's utterance is:"""

run_attribute("Emotion", EMOTION_LABELS, EMOTION_PROMPT)


────────────────────────────────────────────────────────────
  Attribute : Emotion
  Labels    : ['Positive', 'Negative', 'Neutral/Unclear']
────────────────────────────────────────────────────────────


     GPT: 100%|██████████| 5/5 [00:05<00:00,  1.04s/it]


  GPT done → sample: ['Negative', 'Positive', 'Positive']


  CLAUDE: 100%|██████████| 5/5 [00:09<00:00,  1.88s/it]


  CLAUDE done → sample: ['Negative', 'Positive', 'Positive']


...

  GEMINI done → sample: ['ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\nPlease retry in 27.47174213s. [links {\n  description: "Learn more about Gemini API quotas"\n  url: "https://ai.google.dev/gemini-api/docs/rate-limits"\n}\n, violations {\

DEEPSEEK: 100%|██████████| 5/5 [00:08<00:00,  1.78s/it]


  DEEPSEEK done → sample: ['Negative', 'Positive', 'Positive']


 SEALION: 100%|██████████| 5/5 [00:05<00:00,  1.06s/it]

  SEALION done → sample: ["Okay, let's tackle this. The user wants me to", "Okay, let's tackle this. The user wants me to", "Okay, let's tackle this. The user wants me to"]


## Attribute 5 — Question Type
What is the primary sentence function of the utterance?

In [14]:
QUESTION_TYPE_LABELS = ["Declarative/Statement", "Rhetorical Interrogative", "Yes/No Interrogative"]

QUESTION_TYPE_PROMPT = """\
You are a linguist specialising in colloquial Malay. Your task is to read the Malay sentence below and decide its primary sentence function.
Referring to the following three labels and their definitions to make your decision:
Declarative/Statement: The sentence makes an assertion or conveys information. It describes a situation, states a fact, or expresses a view. It is not structured as a question, even if it ends with a particle.
Rhetorical Interrogative: The sentence is phrased as a question but does not expect a genuine answer from the listener. It is used to make a point, express emotion, or emphasise something — the speaker already implies the answer through the question itself.
Yes/No Interrogative: The sentence is a genuine question that invites the listener to confirm or deny something. The speaker does not already know the answer and is seeking a real yes-or-no response.
Speaker: "{TEXT}"
Given the three labels "Declarative/Statement, Rhetorical Interrogative, Yes/No Interrogative", the most likely, single label of the speaker's utterance is:"""

run_attribute("Question_Type", QUESTION_TYPE_LABELS, QUESTION_TYPE_PROMPT)


────────────────────────────────────────────────────────────
  Attribute : Question_Type
  Labels    : ['Declarative/Statement', 'Rhetorical Interrogative', 'Yes/No Interrogative']
────────────────────────────────────────────────────────────


     GPT: 100%|██████████| 5/5 [00:04<00:00,  1.02it/s]


  GPT done → sample: ['Declarative/Statement', 'Rhetorical Interrogative', 'Rhetorical Interrogative']


  CLAUDE: 100%|██████████| 5/5 [00:08<00:00,  1.71s/it]


  CLAUDE done → sample: ['Declarative/Statement', 'Declarative/Statement', 'Declarative/Statement']


...

  GEMINI done → sample: ['ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\nPlease retry in 40.500799944s. [links {\n  description: "Learn more about Gemini API quotas"\n  url: "https://ai.google.dev/gemini-api/docs/rate-limits"\n}\n, violations {

DEEPSEEK: 100%|██████████| 5/5 [00:10<00:00,  2.09s/it]


  DEEPSEEK done → sample: ['Rhetorical Interrogative', 'Rhetorical Interrogative', 'Rhetorical Interrogative']


 SEALION: 100%|██████████| 5/5 [00:04<00:00,  1.03it/s]

  SEALION done → sample: ["Okay, let's tackle this. The user wants me to", "Okay, let's tackle this. The user wants me to", "Okay, let's tackle this. The user wants me to"]


## Save Results

In [15]:
OUTPUT_CSV = Path("../Final Metrics/round1_1a_predictions.csv")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(results)} rows → {OUTPUT_CSV}")

model_prefixes = [f"{m['name']}_" for m in MODEL_RUNS]
keep_cols = ["Text"] + [c for c in results.columns if any(c.startswith(p) for p in model_prefixes)]
display(results[keep_cols].head(5))

Saved 5 rows → ../Final Metrics/round1_1a_predictions.csv


,Text,gpt_Epistemic_Stance,claude_Epistemic_Stance,gemini_Epistemic_Stance,deepseek_Epistemic_Stance,sealion_Epistemic_Stance,gpt_Particle_Position,claude_Particle_Position,gemini_Particle_Position,deepseek_Particle_Position,...,gpt_Emotion,claude_Emotion,gemini_Emotion,deepseek_Emotion,sealion_Emotion,gpt_Question_Type,claude_Question_Type,gemini_Question_Type,deepseek_Question_Type,sealion_Question_Type
0,Menggatal/Tuaran no electric since 8:50pm. Ada ke notis dari tadi sy check fb sesb tda notis. Tu karen kan cukup2 ut...,Uncertain,Certain,"ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this...",Certain,Certain,Middle/End,Middle/End,"ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this...",Middle/End,...,Negative,Negative,"ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this...",Negative,"Okay, let's tackle this. The user wants me to",Declarative/Statement,Declarative/Statement,"ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this...",Rhetorical Interrogative,"Okay, let's tackle this. The user wants me to"
1,Like tak pelik ke nape aku eja lain macam. Dia cakap bahasa wechat. Tu kan nama aku diterbalikkan 😭😂,Uncertain,Certain,"ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this...",Certain,Certain,Middle/End,Middle/End,"ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this...",Middle/End,...,Positive,Positive,"ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this...",Positive,"Okay, let's tackle this. The user wants me to",Rhetorical Interrogative,Declarative/Statement,"ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this...",Rhetorical Interrogative,"Okay, let's tackle this. The user wants me to"
2,Like tak pelik ke nape aku eja lain macam. Dia cakap bahasa wechat. Tu [___] nama aku diterbalikkan 😭😂,Uncertain,Certain,"ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this...",Certain,Certain,N/A,N/A,"ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this...",Middle/End,...,Positive,Positive,"ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this...",Positive,"Okay, let's tackle this. The user wants me to",Rhetorical Interrogative,Declarative/Statement,"ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this...",Rhetorical Interrogative,"Okay, let's tackle this. The user wants me to"
3,"@Kyo_Cleo Pelik [___],puasa ni setan kena ikat,tapi mcm ada yg terlepas je🤣",Uncertain,Uncertain,"ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this...",Certain,Certain,Middle/End,N/A,"ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this...",N/A,...,Positive,Positive,"ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this...",Positive,"Okay, let's tackle this. The user wants me to",Rhetorical Interrogative,Declarative/Statement,"ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this...",Declarative/Statement,"Okay, let's tackle this. The user wants me to"
4,@kikinininirur Kau tak rasis??hahaha penipuan terbesar nak kata diri tak rasis pada orang melayu padahal apa yg dokt...,Certain,Certain,"ERROR: 429 You exceeded your current quota, please check y

## Accuracy Report

Per-attribute accuracy for each model, plus an overall macro-average.

In [ ]:
ATTRIBUTE_LABEL_MAP = {
    "Epistemic_Stance": ["Certain", "Uncertain", "Neutral/Unclear", "Neutral / NA"],
    "Particle_Position": ["Front", "Middle/End", "N/A"],
    "Listener_Agreement": ["Assumed Agreement", "Confirmation Seeking", "Neutral/Unclear"],
    "Emotion": ["Positive", "Negative", "Neutral/Unclear"],
    "Question_Type": ["Declarative/Statement", "Rhetorical Interrogative", "Yes/No Interrogative"],
}

# Malay aliases are mapped back to canonical labels for fair scoring vs ground truth.
ATTRIBUTE_ALIASES = {
    "Epistemic_Stance": {
        "pasti": "Certain",
        "tidak pasti": "Uncertain",
        "neutral/na": "Neutral/Unclear",
        "neutral / na": "Neutral/Unclear",
        "neutral": "Neutral/Unclear",
    },
    "Particle_Position": {
        "hadapan": "Front",
        "depan": "Front",
        "tengah/akhir": "Middle/End",
        "tengah atau akhir": "Middle/End",
        "tiada": "N/A",
        "tidak ada": "N/A",
    },
    "Listener_Agreement": {
        "anggapan persetujuan": "Assumed Agreement",
        "persetujuan diandaikan": "Assumed Agreement",
        "mencari pengesahan": "Confirmation Seeking",
        "neutral/tidak jelas": "Neutral/Unclear",
        "neutral": "Neutral/Unclear",
    },
    "Emotion": {
        "positif": "Positive",
        "negatif": "Negative",
        "neutral/tidak jelas": "Neutral/Unclear",
        "neutral": "Neutral/Unclear",
    },
    "Question_Type": {
        "deklaratif/pernyataan": "Declarative/Statement",
        "soalan retorik": "Rhetorical Interrogative",
        "tanya jawab retorik": "Rhetorical Interrogative",
        "soalan ya/tidak": "Yes/No Interrogative",
        "tanya jawab ya/tidak": "Yes/No Interrogative",
    },
}

EVAL_VARIANTS = {
    "EN": "",
    "MS": "_ms",
}

def normalize(val, label_set, attribute=None):
    if pd.isna(val):
        return val
    raw = str(val).strip()
    v = raw.lower()

    if v in {"na", "n/a", "n.a.", "none", "tiada", "tidak ada"} and "N/A" in label_set:
        return "N/A"

    for lbl in sorted(label_set, key=len, reverse=True):
        if lbl.lower() == v:
            return lbl

    if attribute in ATTRIBUTE_ALIASES:
        for alias, canonical in ATTRIBUTE_ALIASES[attribute].items():
            if alias == v:
                return canonical

    return raw

# ── Per-attribute accuracy table by prompt variant ───────────────────────────
rows = []
model_names = [m["name"] for m in MODEL_RUNS]

for variant_name, suffix in EVAL_VARIANTS.items():
    for attr, labels in ATTRIBUTE_LABEL_MAP.items():
        gt = results[attr].apply(lambda x: normalize(x, labels, attr))
        for model in model_names:
            col = f"{model}_{attr}{suffix}"
            if col not in results.columns:
                continue
            pred = results[col].apply(lambda x: normalize(x, labels, attr))
            acc = (gt == pred).mean()
            rows.append({
                "Variant": variant_name,
                "Attribute": attr,
                "Model": model.upper(),
                "Accuracy": round(acc, 4),
            })

acc_df = pd.DataFrame(rows)
if acc_df.empty:
    print("No prediction columns found for evaluation.")
else:
    print("=" * 55)
    print("  Test 1a — Attribute Accuracy")
    print("=" * 55)
    for variant_name in EVAL_VARIANTS:
        subset = acc_df[acc_df["Variant"] == variant_name]
        if subset.empty:
            continue
        pivot = subset.pivot(index="Attribute", columns="Model", values="Accuracy")
        macro = pivot.mean().rename("MACRO AVG")
        pivot = pd.concat([pivot, macro.to_frame().T])
        pivot.columns.name = None
        pivot.index.name = "Attribute"
        print(f"\n[{variant_name}] Prompt Suite")
        display((pivot * 100).round(1).astype(str) + "%")

  Test 1a — Attribute Accuracy


,CLAUDE,DEEPSEEK,GEMINI,GPT,SEALION
Attribute,,,,,
Emotion,40.0%,40.0%,0.0%,40.0%,0.0%
Epistemic_Stance,80.0%,100.0%,0.0%,20.0%,100.0%
Listener_Agreement,60.0%,40.0%,0.0%,40.0%,0.0%
Particle_Position,60.0%,60.0%,0.0%,40.0%,0.0%
Question_Type,40.0%,60.0%,0.0%,60.0%,0.0%
MACRO AVG,56.0%,60.0%,0.0%,40.0%,20.0%


In [ ]:
from sklearn.metrics import classification_report

# ── Per-class precision / recall / F1 for each attribute × model × variant ───
model_names = [m["name"] for m in MODEL_RUNS]

for variant_name, suffix in EVAL_VARIANTS.items():
    has_any_variant_cols = any(
        f"{model}_{attr}{suffix}" in results.columns
        for model in model_names
        for attr in ATTRIBUTE_LABEL_MAP
    )
    if not has_any_variant_cols:
        continue

    print(f"\n{'='*60}")
    print(f"  Variant: {variant_name}")
    print(f"{'='*60}")

    for attr, labels in ATTRIBUTE_LABEL_MAP.items():
        gt_raw = results[attr].apply(lambda x: normalize(x, labels, attr)).fillna("N/A").astype(str)
        print(f"\n{'━'*60}")
        print(f"  {attr}")
        print(f"{'━'*60}")
        for model in model_names:
            col = f"{model}_{attr}{suffix}"
            if col not in results.columns:
                continue
            pred_raw = results[col].apply(lambda x: normalize(x, labels, attr)).fillna("N/A").astype(str)
            print(f"\n  ── {model.upper()} ──")
            print(classification_report(gt_raw, pred_raw, zero_division=0))


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Epistemic_Stance
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ── GPT ──
              precision    recall  f1-score   support

     Certain       1.00      0.20      0.33         5
   Uncertain       0.00      0.00      0.00         0

    accuracy                           0.20         5
   macro avg       0.50      0.10      0.17         5
weighted avg       1.00      0.20      0.33         5


  ── CLAUDE ──
              precision    recall  f1-score   support

     Certain       1.00      0.80      0.89         5
   Uncertain       0.00      0.00      0.00         0

    accuracy                           0.80         5
   macro avg       0.50      0.40      0.44         5
weighted avg       1.00      0.80      0.89         5


  ── GEMINI ──
                                                                                                                                                               

In [ ]:
# Malay prompt set (same five attributes), evaluated into *_ms columns.

EPISTEMIC_LABELS_MS = ["Pasti", "Tidak Pasti", "Neutral/NA"]
PARTICLE_POSITION_LABELS_MS = ["Hadapan", "Tengah/Akhir", "Tidak Ada"]
LISTENER_LABELS_MS = ["Anggapan Persetujuan", "Mencari Pengesahan", "Neutral/Tidak Jelas"]
EMOTION_LABELS_MS = ["Positif", "Negatif", "Neutral/Tidak Jelas"]
QUESTION_TYPE_LABELS_MS = ["Deklaratif/Pernyataan", "Tanya Jawab Retorik", "Tanya Jawab Ya/Tidak"]

EPISTEMIC_PROMPT_MS = """\
Anda seorang ahli bahasa yang pakar dalam bahasa Melayu kolokial. Tugas anda adalah untuk membaca ayat Bahasa Melayu di bawah dan menentukan sejauh mana kepastian penutur tentang maklumat yang mereka sampaikan.
Rujuk tiga label berikut dan definisinya untuk membuat keputusan anda:
Pasti: Penutur menganggap pernyataan itu sebagai benar atau sah. Tiada lindung nilai, tiada keraguan, dan tiada semakan. Penutur menegaskan maklumat tersebut dengan penuh keyakinan.
Tidak Pasti: Penutur kedengaran tidak pasti, sedang meneka, sedang menganggarkan, atau sedang menyemak sama ada sesuatu itu benar. Perkataan seperti "agaknya" (saya fikir/mungkin), "kot" (mungkin), atau partikel soalan yang menyiasat untuk pengesahan ialah isyarat tipikal.
Neutral/NA: Ayat ini tidak membawa sebarang isyarat kepastian yang boleh dikesan dalam mana-mana arah. Ini terpakai kepada penerangan, arahan, atau ayat neutral apabila kepastian tidak relevan.
Penutur: "{TEXT}"
Memandangkan tiga label "Pasti, Tidak Pasti, Neutral/NA", label tunggal yang paling mungkin untuk ujaran penutur ialah:"""

PARTICLE_POSITION_PROMPT_MS = """\
Anda seorang ahli bahasa yang pakar dalam bahasa Melayu kolokial. Tugas anda adalah untuk membaca ayat Bahasa Melayu di bawah dan menentukan di mana partikel wacana muncul di dalamnya.
Rujuk tiga label berikut dan definisinya untuk membuat keputusan anda:
Hadapan: Partikel itu muncul pada permulaan ayat, sebelum sebarang perkataan kandungan yang lain.
Tengah/Akhir: Partikel itu muncul di mana-mana sahaja selain bahagian hadapan, termasuk pertengahan ayat, sebelum perkataan terakhir, atau di hujung ayat.
Tidak Ada: Tiada partikel wacana dalam ayat (contohnya slot partikel ditunjukkan sebagai "[___]" atau ayat memang tidak mengandungi partikel).
Penutur: "{TEXT}"
Memandangkan tiga label "Hadapan, Tengah/Akhir, Tidak Ada", label tunggal yang paling mungkin untuk ujaran penutur ialah:"""

LISTENER_PROMPT_MS = """\
Anda seorang ahli bahasa yang pakar dalam bahasa Melayu kolokial. Tugas anda adalah untuk membaca ayat Bahasa Melayu di bawah dan memutuskan bagaimana penutur memberi orientasi kepada pendengar dari segi pengetahuan bersama atau persetujuan.
Rujuk tiga label berikut dan definisinya untuk membuat keputusan anda:
Anggapan Persetujuan: Penutur menganggap maklumat tersebut telah dikongsi atau sudah jelas kepada pendengar. Ayat dibentangkan sebagai titik persamaan, tanpa permintaan pengesahan yang eksplisit.
Mencari Pengesahan: Penutur sedang menyemak sama ada pendengar bersetuju, tahu, atau boleh mengesahkan maklumat tersebut. Ayat menjemput atau meminta pengesahan pendengar.
Neutral/Tidak Jelas: Ayat tidak menunjukkan orientasi yang jelas terhadap persetujuan pendengar. Ini terpakai kepada pernyataan biasa, arahan, atau kes yang samar-samar.
Penutur: "{TEXT}"
Memandangkan tiga label "Anggapan Persetujuan, Mencari Pengesahan, Neutral/Tidak Jelas", label tunggal yang paling mungkin untuk ujaran penutur ialah:"""

EMOTION_PROMPT_MS = """\
Anda seorang ahli bahasa yang pakar dalam bahasa Melayu kolokial. Tugas anda adalah untuk membaca ayat Bahasa Melayu di bawah dan menentukan nada emosi penutur.
Rujuk tiga label berikut dan definisinya untuk membuat keputusan anda:
Positif: Penutur meluahkan kegembiraan, keterujaan, semangat, kepuasan, humor, kasih sayang, kelegaan, atau perasaan positif yang lain.
Negatif: Penutur meluahkan kekecewaan, kegusaran, kesedihan, kemarahan, kepahitan, atau perasaan negatif yang lain.
Neutral/Tidak Jelas: Ayat tidak membawa cas emosi yang boleh dikesan dalam kedua-dua arah, atau emosi benar-benar samar-samar.
Penutur: "{TEXT}"
Memandangkan tiga label "Positif, Negatif, Neutral/Tidak Jelas", label tunggal yang paling mungkin untuk ujaran penutur ialah:"""

QUESTION_TYPE_PROMPT_MS = """\
Anda seorang ahli bahasa yang pakar dalam bahasa Melayu kolokial. Tugas anda adalah untuk membaca ayat Bahasa Melayu di bawah dan menentukan fungsi ayat utamanya.
Rujuk tiga label berikut dan definisinya untuk membuat keputusan anda:
Deklaratif/Pernyataan: Ayat membuat penegasan atau menyampaikan maklumat. Ia bukan soalan tulen walaupun mungkin berakhir dengan partikel.
Tanya Jawab Retorik: Ayat berbentuk soalan tetapi tidak mengharapkan jawapan tulen daripada pendengar; ia digunakan untuk menegaskan poin atau meluahkan emosi.
Tanya Jawab Ya/Tidak: Ayat ialah soalan tulen yang meminta pengesahan atau penafian dan mengharapkan jawapan ya/tidak.
Penutur: "{TEXT}"
Memandangkan tiga label "Deklaratif/Pernyataan, Tanya Jawab Retorik, Tanya Jawab Ya/Tidak", label tunggal yang paling mungkin untuk ujaran penutur ialah:"""

MALAY_PROMPT_RUNS = [
    ("Epistemic_Stance", EPISTEMIC_LABELS_MS, EPISTEMIC_PROMPT_MS),
    ("Particle_Position", PARTICLE_POSITION_LABELS_MS, PARTICLE_POSITION_PROMPT_MS),
    ("Listener_Agreement", LISTENER_LABELS_MS, LISTENER_PROMPT_MS),
    ("Emotion", EMOTION_LABELS_MS, EMOTION_PROMPT_MS),
    ("Question_Type", QUESTION_TYPE_LABELS_MS, QUESTION_TYPE_PROMPT_MS),
]

for attr, labels, prompt in MALAY_PROMPT_RUNS:
    run_attribute(attr, labels, prompt, col_suffix="_ms")